# Day 1 · Module 6: Evaluation Loops (Tests-from-Criteria, Coverage Matrix, Vacuous-Test Trap)

**Objective:** generate tests from acceptance criteria instead of from the implementation, build a coverage matrix that maps each criterion to a test and a verified/vacuous verdict, and run one revision loop that records residual risk instead of chasing an all-green suite.


## Relevant Claude Code Docs
Review these before starting the module:
- [Code Review](https://code.claude.com/docs/en/code-review)
- [Find bugs with ultrareview](https://code.claude.com/docs/en/ultrareview)
- [Checkpointing](https://code.claude.com/docs/en/checkpointing)

## 1 · The idea

It is tempting to write tests by reading the code and asserting whatever it currently does. Those tests pass immediately and stay green forever — because they were derived from the implementation's actual behavior, including its blind spots. If the implementation never reads a column, a test that only manipulates that column will still pass, having verified nothing about the behavior it claims to cover.

The fix is to generate tests from the acceptance criteria, not from the code, and to read the code only enough to make the tests compile. Then a critic builds a coverage matrix: for each criterion, which test claims to cover it, and whether that test actually verifies the behavior, verifies it vacuously (passes without exercising the real mechanism), or leaves it untested. A trustworthy evaluation loop ends with that matrix naming at least one criterion that is not truly verified, plus the residual risk the team is choosing to accept — not with "all green."


### Grounding

`scenarios/m6-key-ttl.md` asks for tests that prove `accounts.py`'s `is_api_key_valid(conn, token, at=None, ttl_days=30)` actually expires API keys 30 days after last use. Reading the function's shape:

```python
def is_api_key_valid(conn, token, at=None, ttl_days: int = 30) -> bool:
    row = conn.execute(
        "SELECT last_used_at FROM api_keys WHERE token = ?", (token,)
    ).fetchone()
    ...
    last_used = models.parse(row["last_used_at"])
    return at - last_used <= timedelta(days=ttl_days)
```

it reads only the `last_used_at` column. The `api_keys` table also has a `created_at` column — set when the key is made, never read by this function. A test that backdates `created_at` and calls it "an expiry test" would pass today, would still pass if `is_api_key_valid` were rewritten to ignore last-use entirely, and would have verified nothing about expiry. That is the vacuous-test trap this module is built around: two timestamp columns exist, only one drives the behavior under test, and a test that touches the wrong one still shows a green checkmark.


### Setup check

Run the cell below once per session. It makes sure `contoso` is importable and prints the resolved project root — you'll need `proj` for the exercise and the self-check.


In [ ]:
import subprocess, sys, os
# Ensure src is importable and the sandbox is healthy before you start.
os.chdir(os.path.dirname(os.getcwd())) if os.path.basename(os.getcwd()) in ("day1","day2") else None
root = subprocess.run(["git","rev-parse","--show-toplevel"], capture_output=True, text=True)
proj = root.stdout.strip() or os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
sys.path.insert(0, os.path.join(proj, "src"))
print("project root:", proj)
try:
    import contoso
    print("contoso imported OK")
except Exception as e:
    print("Could not import contoso:", e)
    print("Run `uv sync` in the repo root, then restart the kernel.")


With `proj` resolved, run the trap live: a key with an ancient `created_at` but a fresh `last_used_at` is still `True` (valid) — because expiry only ever looks at `last_used_at`, which the check below confirms by moving the clock forward 31 days:


In [ ]:
import importlib, contoso.db as db, contoso.accounts as accounts
from contoso.models import now, iso
from datetime import timedelta
c = db.connect(":memory:"); db.reset(c)
acct = accounts.create_account(c, "x@x.com"); tok = accounts.create_api_key(c, acct)
# A test that only backdates created_at would still see this as valid:
c.execute("UPDATE api_keys SET created_at = ? WHERE token = ?",
          (iso(now()-timedelta(days=999)), tok)); c.commit()
print("valid despite ancient created_at:", accounts.is_api_key_valid(c, tok))  # True -> trap
print("only last_used_at drives expiry:", accounts.is_api_key_valid(c, tok, at=now()+timedelta(days=31)))  # False


## 2 · Your exercise

Work through these steps in order:

1. Copy `reference/m6/plan.md` to `artifacts/m6/plan.md` — this is your canned technical plan, complete with acceptance criteria (AC1, AC2, AC3) for the 30-day expiry behavior. Treat it as the source of truth for what to test; do not derive tests from `accounts.py` itself.
2. Complete `.claude/agents/tester.md` from `starter-agents/tester.skeleton.md`. Its rules must be explicit that every test case traces to an acceptance criterion in `artifacts/m6/plan.md`, and that the implementation may be read only to make tests compile — never as the source of what to assert.
3. Run an explorer-critic pair over the tester's output. You can reuse the completed agents at `reference/m5/explorer.md` and `reference/m5/critic.md` (copy them into `.claude/agents/` and point them at this ticket), or write your own from the skeletons in `starter-agents/`.
4. Have the critic emit a coverage matrix: one row per acceptance criterion, naming the test that claims to cover it and a verdict of verified yes, verified no, or vacuous. A test that backdates `created_at` and claims to cover expiry must be flagged vacuous, not yes.
5. Feed the critic's blockers through exactly one revision loop — the tester gets one chance to fix a vacuous or missing test, then the loop closes regardless of outcome.
6. Record the before/after coverage matrix in `artifacts/m6/eval-loop.md`, along with the residual risk you are choosing to accept if any criterion still isn't truly verified after the revision loop.


### Coverage matrix skeleton

Copy this table into `artifacts/m6/eval-loop.md` and have the critic fill in the Verdict and Note columns — every participant's matrix should be directly comparable against this same skeleton:

| Criterion | Test | Verdict {verified / vacuous / untested} | Note |
| --- | --- | --- | --- |
| AC1: key invalid at 31 days after last use | `test_...` | | |
| AC2: key valid at 29 days after last use | `test_...` | | |
| AC3: last use resets the 30-day window | `test_...` | | |
| Boundary: exactly 30 days after last use | `test_...` | | |


### What good looks like

`artifacts/m6/eval-loop.md` names at least one acceptance criterion that is NOT truly verified — vacuous or untested — after the revision loop, and it records the residual risk that leaves in plain language. "All green" is not the goal here and should make you suspicious, not satisfied: for this ticket, the boundary case (exactly 30 days) and the reset-on-use behavior (AC3) are the criteria most likely to end up under-tested even after one revision pass.

The failure mode to watch for is a test that backdates `created_at`, asserts `is_api_key_valid` still returns something reasonable, and gets marked "verified" in the matrix for an expiry criterion. That test exercises no part of the TTL logic — it would keep passing even if `is_api_key_valid` were rewritten to ignore `last_used_at` altogether. A coverage matrix that calls that vacuous is doing its job; one that calls it verified has been fooled by the same trap the grounding cell just demonstrated.

Common failure modes:
- A coverage matrix with every row marked "verified" — a suspicious signal on a scenario built specifically to contain a vacuous-test trap.
- A test suite that is green because it asserts on `created_at` rather than `last_used_at`, and a critic that doesn't catch it.
- A revision loop that keeps iterating until everything is green instead of stopping after one pass and recording residual risk.


### Verify your work

This checklist is advisory, not a gate — it reflects back what it finds in `.claude/agents/tester.md` and `artifacts/m6/eval-loop.md`. It's safe to run at any point, including before you've built anything.


In [ ]:
import pathlib, re
tester = pathlib.Path(proj) / ".claude" / "agents" / "tester.md"
print(f"[{'x' if tester.exists() else ' '}] tester.md exists")
ev = pathlib.Path(proj) / "artifacts" / "m6" / "eval-loop.md"
if ev.exists():
    t = ev.read_text()
    tl = t.lower()
    print(f"[x] eval-loop.md present ({len(t.split())} words)")
    has_matrix = "matrix" in tl or "criterion" in tl
    print(f"  [{'x' if has_matrix else ' '}] includes a coverage matrix (criterion->test->verified)?")
    # A trustworthy loop must record an actual matrix ROW whose Verdict column
    # reads vacuous/untested - not merely the word "risk" appearing somewhere
    # in the file. Match a markdown table row like "| ... | vacuous | ... |".
    row_pattern = re.compile(
        r"^\s*\|.*\|\s*(vacuous|untested)\s*\|", re.IGNORECASE | re.MULTILINE
    )
    caught_trap = bool(row_pattern.search(t))
    print(f"  [{'x' if caught_trap else ' '}] a matrix ROW records a vacuous/untested verdict "
          f"(not merely the word \"risk\" somewhere in the file)?")
    print(f"  [{'x' if 'risk' in tl or 'accepted' in tl else ' '}] records accepted residual risk?")
    if has_matrix and not caught_trap:
        print("  ^ FAIL: an all-green matrix (no vacuous/untested row) is the exact "
              "failure mode this module warns about - it means the trap was not caught.")
else:
    print("[ ] artifacts/m6/eval-loop.md missing")


## 3 · Debrief

- Why generate tests from acceptance criteria instead of from the code — what specifically would a code-derived test suite for `is_api_key_valid` have missed?
- What made the `created_at`-backdating test convincing enough to pass as an expiry test? What would a critic need to check to catch it?
- When is a green test suite lying to you, and what does your coverage matrix need to contain to catch that before it ships?

(Related concept: the capstone assembles exploration, planning, implementation, testing, and critique together on a single unseen ticket.)


---
### Take-home

Tests generated from the implementation inherit the implementation's blind spots — they can only ever confirm what the code already does, not what it was supposed to do. Tests generated from acceptance criteria can actually fail, which is what makes them worth running. A coverage matrix that maps criterion to test to verified/vacuous/untested is what turns "the suite is green" into "here is what we know, here is what we don't, and here is the risk we're accepting" — and that last part, the accepted residual risk, is the honest end state of an evaluation loop, not a failure to reach.
